# Using MMIO (Memory-Mapped I/O)

In [18]:
from pynq import Overlay

# 1. 載入 Overlay (確保 .bit 與 .hwh 檔名相同且在同一目錄)
ol = Overlay("half_adder.bit")

# 2. 透過 ? 查詢可以確認 IP 確實存在，名稱對應你圖中的 myip_0
# ol.myip_0?

# 3. 進行半加器真值表測試
print("--- AXI Half Adder Test ---")
for a in [0, 1]:
    for b in [0, 1]:
        # 將 b 放在 Bit 1, a 放在 Bit 0，組合為二進位 (例如 a=1, b=1 -> 0b11 = 3)
        input_val = (b << 1) | a
        
        # 寫入至 Register 0 (偏移量 0x00)
        ol.myip_0.write(0x00, input_val)
        
        # 從 Register 1 (偏移量 0x04) 讀取硬體運算結果
        res = ol.myip_0.read(0x04)
        
        # 拆解硬體吐回來的數值：Bit 0 是 sum, Bit 1 是 carry
        # output = {30'b0, carry, sum}, 用 mask 的方式取值
        sum_val = res & 0x1
        carry_val = (res >> 1) & 0x1
        # carry_val = bool(res & 0x2)
        
        print(f"輸入 A={a}, B={b} | 硬體計算結果: Sum={sum_val}, Carry={carry_val}")

--- AXI Half Adder Test ---
輸入 A=0, B=0 | 硬體計算結果: Sum=0, Carry=0
輸入 A=0, B=1 | 硬體計算結果: Sum=1, Carry=0
輸入 A=1, B=0 | 硬體計算結果: Sum=1, Carry=0
輸入 A=1, B=1 | 硬體計算結果: Sum=0, Carry=1


# Using Driver (create user-friendly API)

In [23]:
from pynq import MMIO

class MyipDriver:
    """簡單 MMIO-based driver for myip half-adder."""
    def __init__(self, description):
        params = description.get('parameters', {})
        base = description.get('phys_addr') or params.get('C_S00_AXI_BASEADDR')
        if isinstance(base, str):
            base = int(base, 0)
        high = params.get('C_S00_AXI_HIGHADDR')
        if high:
            high = int(high, 0)
            low = int(params.get('C_S00_AXI_BASEADDR'), 0)
            size = high - low + 1
        else:
            size = description.get('addr_range', 4096)
        if base is None:
            raise ValueError("Cannot determine base address for IP.")
        self.mmio = MMIO(int(base), int(size))
        self.OFF_INPUT = 0x00
        self.OFF_OUTPUT = 0x04

    def write_input(self, a, b):
        val = ((b & 0x1) << 1) | (a & 0x1)
        self.mmio.write(self.OFF_INPUT, val)

    def read_output(self):
        res = self.mmio.read(self.OFF_OUTPUT)
        return res & 0x1, (res >> 1) & 0x1

In [24]:
from pynq import Overlay

ol = Overlay("half_adder.bit")
drv = MyipDriver(ol.ip_dict['myip_0'])

print("--- AXI Half Adder Test (driver) ---")
for a in [0, 1]:
    for b in [0, 1]:
        drv.write_input(a, b)
        s, c = drv.read_output()
        print(f"A={a}, B={b} -> Sum={s}, Carry={c}")

--- AXI Half Adder Test (driver) ---
A=0, B=0 -> Sum=0, Carry=0
A=0, B=1 -> Sum=1, Carry=0
A=1, B=0 -> Sum=1, Carry=0
A=1, B=1 -> Sum=0, Carry=1
